# Adım 6 - PySpark MLlib + MLflow

Bu notebook `track_genre` tahmini için 5 sınıflandırma modeli eğitir:

- Logistic Regression
- Decision Tree Classifier
- Random Forest Classifier
- Gradient Boosted Trees (GBT) Classifier
- Naive Bayes

Her model için Accuracy, weighted F1, weighted Precision, weighted Recall, AUC-ROC ve Confusion Matrix hesaplanır. Sonuçlar MLflow'a kaydedilir.


## Windows Kullanım Notu

Bu dosya proje kök klasörü altında çalışacak şekilde hazırlandı. VS Code içinde kernel olarak `Spotify BigData` seç.

MLflow UI için notebook bittikten sonra PowerShell'de:

```powershell
cd C:\Users\HP\Documents\GitHub\spotify-bigdata-project
.\.venv\Scripts\Activate.ps1
mlflow ui --backend-store-uri .\mlruns --host 127.0.0.1 --port 5000
```

Sonra tarayıcıda `http://127.0.0.1:5000` adresini aç.


In [1]:
# Temel kütüphaneler ve proje klasörleri
from pathlib import Path
import json
import math
import os
import shutil
import warnings

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

def find_base_dir() -> Path:
    # Notebook farklı klasörden çalıştırılsa bile proje kökünü buluyoruz.
    current = Path.cwd().resolve()
    for candidate in [current, *current.parents]:
        if (candidate / ".git").exists() or (candidate / "docker-compose.yml").exists():
            return candidate
    if current.name.lower() == "notebooks":
        return current.parent
    return current

BASE_DIR = find_base_dir()
PROJECT_ROOT = BASE_DIR
DASHBOARD_DIR = BASE_DIR / "dashboard"
DASHBOARD_DATA_DIR = DASHBOARD_DIR / "data"
RUN_ARTIFACT_DIR = BASE_DIR / "ml_artifacts"
MLFLOW_TRACKING_DIR = BASE_DIR / "mlruns"

for folder in [DASHBOARD_DIR, DASHBOARD_DATA_DIR, RUN_ARTIFACT_DIR, MLFLOW_TRACKING_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

# ML için önce Kişi 3'ün Feature Engineering çıktısı aranır.
# Kendi veri yolun farklıysa DELTA_TABLE_PATH'i elle değiştirebilirsin.
DELTA_CANDIDATES = [
    PROJECT_ROOT / "delta-lake" / "gold" / "features",
    BASE_DIR / "delta-lake" / "gold" / "features",
    PROJECT_ROOT / "delta-lake" / "silver",
    BASE_DIR / "delta-lake" / "silver",
    BASE_DIR / "data" / "delta" / "spotify_tracks",
    BASE_DIR / "delta" / "spotify_tracks",
    BASE_DIR / "spark" / "data" / "delta" / "spotify_tracks",
    PROJECT_ROOT / "data" / "delta" / "spotify_tracks",
    PROJECT_ROOT / "spark" / "data" / "delta" / "spotify_tracks",
    PROJECT_ROOT / "delta" / "spotify_tracks",
]

DELTA_TABLE_PATH = next(
    (path for path in DELTA_CANDIDATES if (path / "_delta_log").exists()),
    DELTA_CANDIDATES[0],
)

print(f"Proje kökü: {BASE_DIR}")
print(f"Delta tablo yolu: {DELTA_TABLE_PATH}")
print(f"MLflow kayıt klasörü: {MLFLOW_TRACKING_DIR}")


Proje kökü: /home/jovyan
Delta tablo yolu: /home/jovyan/delta-lake/gold/features
MLflow kayıt klasörü: /home/jovyan/mlruns


In [2]:
# Spark ve Delta Lake oturumu
from pyspark.sql import SparkSession, Window, functions as F

try:
    from delta import configure_spark_with_delta_pip
except ModuleNotFoundError:
    configure_spark_with_delta_pip = None

spark_builder = (
    SparkSession.builder
    .appName("SpotifyTrackGenreMLflow")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .config("spark.sql.shuffle.partitions", "4")
    .config("spark.sql.execution.arrow.pyspark.enabled", "true")
    .config("spark.serializer", "org.apache.spark.serializer.KryoSerializer")
    .config("spark.driver.memory", "6g")
    .config("spark.executor.memory", "4g")
    .config("spark.driver.maxResultSize", "2g")
    .config("spark.sql.warehouse.dir", str(BASE_DIR / "spark-warehouse"))
)

if configure_spark_with_delta_pip is not None:
    spark = configure_spark_with_delta_pip(spark_builder).getOrCreate()
else:
    spark = spark_builder.getOrCreate()
spark.sparkContext.setLogLevel("WARN")

print("Spark sürümü:", spark.version)


Spark sürümü: 3.5.0


In [3]:
# Delta Lake tablosunu oku
if not (DELTA_TABLE_PATH / "_delta_log").exists():
    raise FileNotFoundError(
        "Delta tablosu bulunamadı. DELTA_TABLE_PATH değişkenini kendi Delta klasörüne göre düzenle. "
        "Zaman serisi grafiği için bu tabloda Kafka Producer tarafından eklenen timestamp kolonu olmalı."
    )

df_raw = spark.read.format("delta").load(str(DELTA_TABLE_PATH))

print("Ham satır sayısı:", df_raw.count())
df_raw.printSchema()
display(df_raw.limit(5).toPandas())


Ham satır sayısı: 89740
root
 |-- track_id: string (nullable = true)
 |-- track_name: string (nullable = true)
 |-- artists: string (nullable = true)
 |-- album_name: string (nullable = true)
 |-- track_genre: string (nullable = true)
 |-- popularity: integer (nullable = true)
 |-- danceability: float (nullable = true)
 |-- energy: float (nullable = true)
 |-- loudness: float (nullable = true)
 |-- tempo: float (nullable = true)
 |-- speechiness: float (nullable = true)
 |-- acousticness: float (nullable = true)
 |-- instrumentalness: float (nullable = true)
 |-- liveness: float (nullable = true)
 |-- valence: float (nullable = true)
 |-- duration_ms: integer (nullable = true)
 |-- key: integer (nullable = true)
 |-- mode: integer (nullable = true)
 |-- time_signature: integer (nullable = true)
 |-- explicit: integer (nullable = true)
 |-- kafka_timestamp: string (nullable = true)
 |-- user_id: string (nullable = true)
 |-- event_type: string (nullable = true)
 |-- energy_danceability_

,track_id,track_name,artists,album_name,track_genre,popularity,danceability,energy,loudness,tempo,...,time_signature,explicit,kafka_timestamp,user_id,event_type,energy_danceability_ratio,loudness_normalized,tempo_category,energy_acoustic_contrast,dancefloor_score
0,001APMDOl3qtx1526T11n1,Better,Pink Sweat$;Kirby,New RnB,chill,0,0.6130,0.47100,-6.644000,143.063995,...,4,0,2026-05-12T08:30:09.485503,b52bb185-7c53-498a-bf86-21ea6bac6653,track_played,0.768227,0.793278,hizli,0.15500,0.50950
1,002uYDBLOvJz21C4FuArDS,Find Me - Sigma VIP Remix,Sigma;Birdy,Find Me (Remixes),drum-and-bass,20,0.4150,0.88800,-2.544000,174.985992,...,4,0,2026-05-12T08:30:48.163904,c9e5ebcc-93cf-423b-9135-fd446b14eb09,track_played,2.139244,0.869116,hizli,0.88290,0.28050
2,004G9E3EZhxxn5aE9yEQqx,Sandwiches de Miga,Pappo's Blues,"Pappo's Blues, Vol. 3",punk-rock,36,0.3930,0.77900,-8.103000,88.418999,...,4,0,2026-05-12T08:33:31.483209,43ad7111-d8e9-46db-a88b-7c46f84fd250,track_played,1.981684,0.766291,yavas,0.77599,0.45900
3,004iWPkSRbvOEvAPLWHl9M,Mister Love,Ernest Tubb;The Wilburn Brothers,Definitive Hits,honky-tonk,13,0.6380,0.21400,-10.258000,117.764999,...,4,0,2026-05-12T08:31:58.118275,5e201e18-74d2-488a-bca0-969be96a4313,track_played,0.335371,0.726430,orta,-0.61100,0.52150
4,006ATYzgynEKIPgVaT5LQM,528Hz Energía curativa profunda,Mc_team,Frecuencias Curativas Solfeggio 528 Hz,world-music,24,0.0865,0.00504,-36.082001,169.362000,...,4,0,2026-05-12T08:34:49.268443,e49868a0-e44b-4f81-9b80-2fe1862e510b,track_played,0.058199,0.248765,hizli,-0.98596,0.11225


In [4]:
# Feature Engineering çıktısı beklenen kolonlar
label_col = "track_genre"
feature_columns = [
    "danceability",
    "energy",
    "loudness_normalized",
    "speechiness",
    "acousticness",
    "instrumentalness",
    "liveness",
    "valence",
    "tempo_category",
    "energy_acoustic_contrast",
    "dancefloor_score",
    "popularity",
    "key",
    "mode",
    "time_signature",
]
numeric_features = [col for col in feature_columns if col != "tempo_category"]
categorical_features = ["tempo_category"]

df = df_raw

# Önceki adımlarda bazı feature kolonları oluşmadıysa temel yedekleri üretiyoruz.
# Asıl projede bu kolonların Feature Engineering adımından gelmesi beklenir.
if "loudness_normalized" not in df.columns and "loudness" in df.columns:
    bounds = df.agg(
        F.min("loudness").alias("min_loudness"),
        F.max("loudness").alias("max_loudness"),
    ).first()
    min_loudness = float(bounds["min_loudness"] or 0.0)
    max_loudness = float(bounds["max_loudness"] or 1.0)
    denom = max(max_loudness - min_loudness, 1e-9)
    df = df.withColumn(
        "loudness_normalized",
        (F.col("loudness").cast("double") - F.lit(min_loudness)) / F.lit(denom),
    )

if "tempo_category" not in df.columns and "tempo" in df.columns:
    df = df.withColumn(
        "tempo_category",
        F.when(F.col("tempo").cast("double") < 90, F.lit("yavas"))
        .when(F.col("tempo").cast("double") <= 130, F.lit("orta"))
        .otherwise(F.lit("hizli")),
    )

if "energy_acoustic_contrast" not in df.columns:
    df = df.withColumn(
        "energy_acoustic_contrast",
        F.col("energy").cast("double") - F.col("acousticness").cast("double"),
    )

if "dancefloor_score" not in df.columns:
    df = df.withColumn(
        "dancefloor_score",
        (
            F.col("danceability").cast("double")
            + F.col("valence").cast("double")
        ) / F.lit(2.0),
    )

required_cols = [label_col] + feature_columns
missing_cols = [col for col in required_cols if col not in df.columns]
if missing_cols:
    raise ValueError(f"Eksik kolonlar var: {missing_cols}")

# Modelleme için tipleri netleştiriyoruz.
for col_name in numeric_features:
    df = df.withColumn(col_name, F.col(col_name).cast("double"))

df_model = (
    df
    .withColumn(label_col, F.col(label_col).cast("string"))
    .withColumn("tempo_category", F.coalesce(F.col("tempo_category").cast("string"), F.lit("bilinmiyor")))
    .select(label_col, *feature_columns)
    .dropna(subset=[label_col])
)

# Yerel Docker ortamında GBT OneVsRest, 100+ türde belleği zorlayabilir.
# Bu yüzden ML deneyini en sık görülen türlerle ve tür başına dengeli örnekle sınırlıyoruz.
max_genres_for_ml = 25
max_rows_per_genre = 500
top_genres = [
    row[label_col]
    for row in (
        df_model
        .groupBy(label_col)
        .count()
        .orderBy(F.desc("count"))
        .limit(max_genres_for_ml)
        .collect()
    )
]
df_model = (
    df_model
    .where(F.col(label_col).isin(top_genres))
    .withColumn("_genre_row_number", F.row_number().over(Window.partitionBy(label_col).orderBy(F.rand(seed=42))))
    .where(F.col("_genre_row_number") <= max_rows_per_genre)
    .drop("_genre_row_number")
    .cache()
)

print("Modelleme satır sayısı:", df_model.count())
print(f"Modelleme tür sayısı: {len(top_genres)}")
print(f"Tür başına maksimum örnek: {max_rows_per_genre}")
display(df_model.limit(5).toPandas())


Modelleme satır sayısı: 12500
Modelleme tür sayısı: 25
Tür başına maksimum örnek: 500


,track_genre,danceability,energy,loudness_normalized,speechiness,acousticness,instrumentalness,liveness,valence,tempo_category,energy_acoustic_contrast,dancefloor_score,popularity,key,mode,time_signature
0,acoustic,0.607,0.473,0.757931,0.0340,0.712,0.000000,0.6600,0.540,orta,-0.239,0.5735,56.0,2.0,1.0,4.0
1,acoustic,0.571,0.696,0.755470,0.0739,0.140,0.000000,0.0973,0.662,hizli,0.556,0.6165,30.0,9.0,1.0,4.0
2,acoustic,0.458,0.363,0.796441,0.0320,0.892,0.000004,0.1550,0.492,orta,-0.529,0.4750,55.0,5.0,1.0,3.0
3,acoustic,0.522,0.152,0.638792,0.0303,0.788,0.000024,0.1480,0.133,hizli,-0.636,0.3275,60.0,2.0,1.0,4.0
4,acoustic,0.259,0.195,0.663892,0.0331,0.264,0.433000,0.3520,0.183,hizli,-0.069,0.2210,49.0,9.0,0.0,4.0


In [5]:
# Label encoding ve train/test ayrımı
from pyspark.ml.feature import StringIndexer

label_indexer = StringIndexer(inputCol=label_col, outputCol="label", handleInvalid="skip")
label_model = label_indexer.fit(df_model)
df_labeled = label_model.transform(df_model)

label_names = list(label_model.labels)
num_classes = len(label_names)

if num_classes < 2:
    raise ValueError("Sınıflandırma için en az 2 farklı track_genre gerekir.")

train_df, test_df = df_labeled.randomSplit([0.8, 0.2], seed=42)
train_count = train_df.count()
test_count = test_df.count()

label_mapping = pd.DataFrame({
    "label_index": list(range(num_classes)),
    "track_genre": label_names,
})
label_mapping.to_csv(DASHBOARD_DATA_DIR / "label_mapping.csv", index=False, encoding="utf-8-sig")

print(f"Sınıf sayısı: {num_classes}")
print(f"Train satır sayısı: {train_count}")
print(f"Test satır sayısı: {test_count}")
display(label_mapping.head(10))


Sınıf sayısı: 25
Train satır sayısı: 10053
Test satır sayısı: 2447


,label_index,track_genre
0,0,acoustic
1,1,afrobeat
2,2,alt-rock
3,3,ambient
4,4,anime
5,5,black-metal
6,6,bluegrass
7,7,breakbeat
8,8,cantopop
9,9,chicago-house


In [6]:
# Preprocessing pipeline: eksik değer doldurma, tempo kategorisini sayısallaştırma, vektörleştirme ve ölçekleme
from pyspark.ml import Pipeline
from pyspark.ml.feature import Imputer, MinMaxScaler, OneHotEncoder, VectorAssembler

numeric_imputed_features = [f"{col}_imputed" for col in numeric_features]

def build_preprocessing_stages():
    imputer = Imputer(
        inputCols=numeric_features,
        outputCols=numeric_imputed_features,
        strategy="median",
    )

    tempo_indexer = StringIndexer(
        inputCol="tempo_category",
        outputCol="tempo_category_index",
        handleInvalid="keep",
    )

    tempo_encoder = OneHotEncoder(
        inputCols=["tempo_category_index"],
        outputCols=["tempo_category_ohe"],
        dropLast=False,
    )

    assembler_input_cols = [
        f"{col}_imputed" if col in numeric_features else "tempo_category_ohe"
        for col in feature_columns
    ]

    assembler = VectorAssembler(
        inputCols=assembler_input_cols,
        outputCol="raw_features",
        handleInvalid="keep",
    )

    # Naive Bayes negatif değer kabul etmediği için tüm modellerde ortak 0-1 ölçekleme kullanıyoruz.
    scaler = MinMaxScaler(inputCol="raw_features", outputCol="features")

    return [imputer, tempo_indexer, tempo_encoder, assembler, scaler]


In [7]:
# İstenen 5 model
from pyspark.ml.classification import (
    DecisionTreeClassifier,
    GBTClassifier,
    LogisticRegression,
    NaiveBayes,
    OneVsRest,
    RandomForestClassifier,
)

seed = 42

logistic_regression = LogisticRegression(
    featuresCol="features",
    labelCol="label",
    predictionCol="prediction",
    probabilityCol="probability",
    maxIter=60,
    regParam=0.01,
    elasticNetParam=0.0,
    family="multinomial",
)

decision_tree = DecisionTreeClassifier(
    featuresCol="features",
    labelCol="label",
    predictionCol="prediction",
    probabilityCol="probability",
    maxDepth=8,
    impurity="gini",
    seed=seed,
)

random_forest = RandomForestClassifier(
    featuresCol="features",
    labelCol="label",
    predictionCol="prediction",
    probabilityCol="probability",
    numTrees=25,
    maxDepth=8,
    featureSubsetStrategy="sqrt",
    subsamplingRate=0.8,
    seed=seed,
)

gbt_base = GBTClassifier(
    featuresCol="features",
    labelCol="label",
    predictionCol="prediction",
    maxIter=5,
    maxDepth=3,
    stepSize=0.1,
    subsamplingRate=0.8,
    seed=seed,
)

# Spark GBTClassifier doğrudan binary çalışır. track_genre çok sınıflıysa OneVsRest ile çok sınıflı hale getiriyoruz.
if num_classes > 2:
    gbt_classifier = OneVsRest(
        classifier=gbt_base,
        featuresCol="features",
        labelCol="label",
        predictionCol="prediction",
        parallelism=1,
    )
else:
    gbt_classifier = gbt_base

naive_bayes = NaiveBayes(
    featuresCol="features",
    labelCol="label",
    predictionCol="prediction",
    probabilityCol="probability",
    smoothing=1.0,
    modelType="multinomial",
)

model_specs = [
    {
        "name": "Logistic Regression",
        "experiment": "Kisi4_Logistic_Regression",
        "estimator": logistic_regression,
        "params": {
            "maxIter": 60,
            "regParam": 0.01,
            "elasticNetParam": 0.0,
            "family": "multinomial",
        },
    },
    {
        "name": "Decision Tree Classifier",
        "experiment": "Kisi4_Decision_Tree_Classifier",
        "estimator": decision_tree,
        "params": {
            "maxDepth": 8,
            "impurity": "gini",
            "seed": seed,
        },
    },
    {
        "name": "Random Forest Classifier",
        "experiment": "Kisi4_Random_Forest_Classifier",
        "estimator": random_forest,
        "params": {
            "numTrees": 25,
            "maxDepth": 8,
            "featureSubsetStrategy": "sqrt",
            "subsamplingRate": 0.8,
            "seed": seed,
        },
    },
    {
        "name": "Gradient Boosted Trees (GBT) Classifier",
        "experiment": "Kisi4_GBT_Classifier",
        "estimator": gbt_classifier,
        "params": {
            "maxIter": 5,
            "maxDepth": 3,
            "stepSize": 0.1,
            "subsamplingRate": 0.8,
            "seed": seed,
            "multiclass_strategy": "OneVsRest" if num_classes > 2 else "Binary GBTClassifier",
        },
    },
    {
        "name": "Naive Bayes",
        "experiment": "Kisi4_Naive_Bayes",
        "estimator": naive_bayes,
        "params": {
            "smoothing": 1.0,
            "modelType": "multinomial",
        },
    },
]

print("Eğitilecek model sayısı:", len(model_specs))


Eğitilecek model sayısı: 5


In [8]:
# Metrik, ROC, confusion matrix ve feature importance yardımcı fonksiyonları
from sklearn.metrics import accuracy_score, confusion_matrix, precision_recall_fscore_support, roc_auc_score, roc_curve
from sklearn.preprocessing import label_binarize

def safe_name(value: str) -> str:
    return (
        value.lower()
        .replace(" ", "_")
        .replace("(", "")
        .replace(")", "")
        .replace("-", "_")
    )

def vector_to_array(value):
    if value is None:
        return None
    if hasattr(value, "toArray"):
        return np.asarray(value.toArray(), dtype=float)
    if isinstance(value, (list, tuple, np.ndarray)):
        return np.asarray(value, dtype=float)
    return np.asarray([float(value)], dtype=float)

def softmax(matrix: np.ndarray) -> np.ndarray:
    matrix = np.asarray(matrix, dtype=float)
    matrix = np.nan_to_num(matrix, nan=0.0, posinf=0.0, neginf=0.0)
    shifted = matrix - np.max(matrix, axis=1, keepdims=True)
    exp_values = np.exp(shifted)
    denom = np.sum(exp_values, axis=1, keepdims=True)
    denom[denom == 0] = 1.0
    return exp_values / denom

def score_matrix_from_pdf(pdf: pd.DataFrame, score_col: str | None, class_count: int):
    if score_col is None or score_col not in pdf.columns:
        return None
    arrays = [vector_to_array(value) for value in pdf[score_col]]
    arrays = [arr for arr in arrays if arr is not None]
    if not arrays:
        return None
    scores = np.vstack(arrays)
    if scores.shape[1] < class_count:
        scores = np.pad(scores, ((0, 0), (0, class_count - scores.shape[1])), mode="constant")
    if scores.shape[1] > class_count:
        scores = scores[:, :class_count]

    row_sums = scores.sum(axis=1)
    looks_like_probability = (
        np.all(scores >= -1e-9)
        and np.all(scores <= 1 + 1e-9)
        and np.allclose(row_sums, 1.0, atol=1e-3)
    )
    if not looks_like_probability:
        scores = softmax(scores)
    return scores

def compute_auc_and_roc(y_true: np.ndarray, scores: np.ndarray | None, model_name: str, class_count: int):
    if scores is None:
        return float("nan"), pd.DataFrame({"model": [model_name, model_name], "fpr": [0.0, 1.0], "tpr": [0.0, 1.0]})

    try:
        if class_count == 2:
            positive_scores = scores[:, 1]
            auc_value = roc_auc_score(y_true, positive_scores)
            fpr, tpr, _ = roc_curve(y_true, positive_scores, pos_label=1)
        else:
            classes = np.arange(class_count)
            present_classes = np.array(sorted(set(y_true.tolist())))
            if len(present_classes) < 2:
                raise ValueError("Test verisinde AUC için en az 2 sınıf görünmeli.")
            y_bin_all = label_binarize(y_true, classes=classes)
            y_bin = y_bin_all[:, present_classes]
            present_scores = scores[:, present_classes]
            auc_value = roc_auc_score(y_bin, present_scores, average="weighted")
            fpr, tpr, _ = roc_curve(y_bin.ravel(), present_scores.ravel())
    except ValueError as error:
        print(f"{model_name} için AUC hesaplanamadı: {error}")
        auc_value = float("nan")
        fpr, tpr = np.array([0.0, 1.0]), np.array([0.0, 1.0])

    if len(fpr) > 1000:
        idx = np.linspace(0, len(fpr) - 1, 1000).astype(int)
        fpr, tpr = fpr[idx], tpr[idx]

    roc_df = pd.DataFrame({"model": model_name, "fpr": fpr, "tpr": tpr})
    return float(auc_value), roc_df

def evaluate_predictions(predictions, model_name: str):
    score_col = None
    if "probability" in predictions.columns:
        score_col = "probability"
    elif "rawPrediction" in predictions.columns:
        score_col = "rawPrediction"

    select_cols = ["label", "prediction"]
    if score_col:
        select_cols.append(score_col)

    pred_pdf = predictions.select(*select_cols).toPandas()
    pred_pdf["label"] = pred_pdf["label"].astype(int)
    pred_pdf["prediction"] = pred_pdf["prediction"].astype(int)

    y_true = pred_pdf["label"].to_numpy()
    y_pred = pred_pdf["prediction"].to_numpy()

    precision_value, recall_value, f1_value, _ = precision_recall_fscore_support(
        y_true,
        y_pred,
        average="weighted",
        zero_division=0,
    )

    metrics = {
        "accuracy": accuracy_score(y_true, y_pred),
        "f1_weighted": f1_value,
        "precision_weighted": precision_value,
        "recall_weighted": recall_value,
    }

    scores = score_matrix_from_pdf(pred_pdf, score_col, num_classes)
    auc_value, roc_df = compute_auc_and_roc(y_true, scores, model_name, num_classes)
    metrics["auc_roc"] = auc_value

    cm = confusion_matrix(y_true, y_pred, labels=list(range(num_classes)))
    return metrics, cm, roc_df, pred_pdf

def feature_names_from_pipeline_model(pipeline_model):
    # Pipeline sırası: Imputer, StringIndexer, OneHotEncoder, VectorAssembler, MinMaxScaler, Model
    tempo_model = pipeline_model.stages[1]
    encoder_model = pipeline_model.stages[2]
    tempo_labels = list(getattr(tempo_model, "labels", []))

    try:
        tempo_size = int(encoder_model.categorySizes[0])
    except Exception:
        tempo_size = len(tempo_labels)

    while len(tempo_labels) < tempo_size:
        tempo_labels.append("bilinmeyen")

    readable_tempo = [f"tempo_category={name}" for name in tempo_labels[:tempo_size]]
    readable_features = []
    for feature_name in feature_columns:
        if feature_name == "tempo_category":
            readable_features.extend(readable_tempo)
        else:
            readable_features.append(feature_name)
    return readable_features

def tree_importance_dataframe(pipeline_model, model_name: str):
    classifier_model = pipeline_model.stages[-1]

    if hasattr(classifier_model, "featureImportances"):
        values = classifier_model.featureImportances.toArray()
    elif hasattr(classifier_model, "models"):
        # OneVsRest GBT için her sınıfın binary GBT önem skorlarını ortalıyoruz.
        arrays = [
            model.featureImportances.toArray()
            for model in classifier_model.models
            if hasattr(model, "featureImportances")
        ]
        if not arrays:
            return None
        values = np.mean(np.vstack(arrays), axis=0)
    else:
        return None

    names = feature_names_from_pipeline_model(pipeline_model)
    length = min(len(names), len(values))
    importance_df = pd.DataFrame({
        "model": model_name,
        "feature": names[:length],
        "importance": values[:length],
    }).sort_values("importance", ascending=False)
    return importance_df


In [9]:
# MLflow ile modelleri eğit, metrikleri logla ve dashboard ara dosyalarını üret
import mlflow
import mlflow.spark

mlflow.set_tracking_uri(MLFLOW_TRACKING_DIR.as_uri())

all_metrics = []
all_roc_frames = []
best_result = {
    "f1_weighted": -1.0,
    "model_name": None,
    "predictions": None,
    "confusion_matrix": None,
}

for spec in model_specs:
    model_name = spec["name"]
    print("\n" + "=" * 80)
    print(f"Model eğitiliyor: {model_name}")

    pipeline = Pipeline(stages=build_preprocessing_stages() + [spec["estimator"]])
    mlflow.set_experiment(spec["experiment"])

    with mlflow.start_run(run_name=model_name) as run:
        # Parametreleri MLflow'a yazıyoruz.
        mlflow.log_param("model_name", model_name)
        mlflow.log_param("target_column", label_col)
        mlflow.log_param("feature_columns", ",".join(feature_columns))
        mlflow.log_param("num_classes", num_classes)
        mlflow.log_param("train_count", train_count)
        mlflow.log_param("test_count", test_count)
        mlflow.log_param("delta_table_path", str(DELTA_TABLE_PATH))
        for param_name, param_value in spec["params"].items():
            mlflow.log_param(param_name, param_value)

        fitted_pipeline = pipeline.fit(train_df)
        predictions = fitted_pipeline.transform(test_df)

        metrics, cm, roc_df, pred_pdf = evaluate_predictions(predictions, model_name)

        # Metrikleri MLflow'a yazıyoruz.
        for metric_name, metric_value in metrics.items():
            if metric_value is not None and not math.isnan(float(metric_value)):
                mlflow.log_metric(metric_name, float(metric_value))

        run_dir = RUN_ARTIFACT_DIR / safe_name(model_name)
        if run_dir.exists():
            shutil.rmtree(run_dir)
        run_dir.mkdir(parents=True, exist_ok=True)

        cm_df = pd.DataFrame(cm, index=label_names, columns=label_names)
        cm_path = run_dir / "confusion_matrix.csv"
        cm_df.to_csv(cm_path, encoding="utf-8-sig")
        mlflow.log_artifact(str(cm_path), artifact_path="evaluation")

        metrics_path = run_dir / "metrics.json"
        with metrics_path.open("w", encoding="utf-8") as file:
            json.dump(metrics, file, ensure_ascii=False, indent=2)
        mlflow.log_artifact(str(metrics_path), artifact_path="evaluation")

        importance_df = None
        if model_name in ["Random Forest Classifier", "Gradient Boosted Trees (GBT) Classifier"]:
            importance_df = tree_importance_dataframe(fitted_pipeline, model_name)
            if importance_df is not None and not importance_df.empty:
                if model_name == "Random Forest Classifier":
                    importance_path = DASHBOARD_DATA_DIR / "rf_feature_importance.csv"
                else:
                    importance_path = DASHBOARD_DATA_DIR / "gbt_feature_importance.csv"

                importance_df.to_csv(importance_path, index=False, encoding="utf-8-sig")
                mlflow.log_artifact(str(importance_path), artifact_path="feature_importance")
                mlflow.log_param("top_feature", importance_df.iloc[0]["feature"])
                mlflow.log_metric("top_feature_importance", float(importance_df.iloc[0]["importance"]))

                print("En önemli 10 özellik:")
                display(importance_df.head(10))

        mlflow.spark.log_model(fitted_pipeline, artifact_path="spark_model")

        row = {
            "model": model_name,
            "experiment": spec["experiment"],
            "run_id": run.info.run_id,
            **metrics,
        }
        all_metrics.append(row)
        all_roc_frames.append(roc_df)

        print("Metrikler:")
        display(pd.DataFrame([row]))
        print("Confusion Matrix ilk 10x10 görünüm:")
        display(cm_df.iloc[:10, :10])

        if metrics["f1_weighted"] > best_result["f1_weighted"]:
            best_result = {
                "f1_weighted": metrics["f1_weighted"],
                "model_name": model_name,
                "predictions": pred_pdf.copy(),
                "confusion_matrix": cm_df.copy(),
            }

        predictions.unpersist()

metrics_df = pd.DataFrame(all_metrics).sort_values("f1_weighted", ascending=False)
metrics_df.to_csv(DASHBOARD_DATA_DIR / "model_metrics.csv", index=False, encoding="utf-8-sig")

roc_all_df = pd.concat(all_roc_frames, ignore_index=True)
roc_all_df.to_csv(DASHBOARD_DATA_DIR / "roc_curve_points.csv", index=False, encoding="utf-8-sig")

best_predictions = best_result["predictions"].copy()
best_predictions["actual_genre"] = best_predictions["label"].map(lambda idx: label_names[int(idx)])
best_predictions["predicted_genre"] = best_predictions["prediction"].map(lambda idx: label_names[int(idx)] if int(idx) < len(label_names) else "bilinmeyen")
best_predictions[["label", "prediction", "actual_genre", "predicted_genre"]].to_csv(
    DASHBOARD_DATA_DIR / "best_model_predictions.csv",
    index=False,
    encoding="utf-8-sig",
)

best_result["confusion_matrix"].to_csv(
    DASHBOARD_DATA_DIR / "best_confusion_matrix.csv",
    encoding="utf-8-sig",
)

best_metadata = {
    "best_model_name": best_result["model_name"],
    "selection_metric": "f1_weighted",
    "best_f1_weighted": best_result["f1_weighted"],
}
with (DASHBOARD_DATA_DIR / "best_model_metadata.json").open("w", encoding="utf-8") as file:
    json.dump(best_metadata, file, ensure_ascii=False, indent=2)

print("\nEğitim tamamlandı.")
print("En iyi model:", best_result["model_name"])
display(metrics_df)


2026/05/12 20:19:15 INFO mlflow.tracking.fluent: Experiment with name 'Kisi4_Logistic_Regression' does not exist. Creating a new experiment.



Model eğitiliyor: Logistic Regression
Metrikler:


,model,experiment,run_id,accuracy,f1_weighted,precision_weighted,recall_weighted,auc_roc
0,Logistic Regression,Kisi4_Logistic_Regression,0e0b046798774260a018c23786f200e3,0.466285,0.449011,0.466838,0.466285,0.919739


Confusion Matrix ilk 10x10 görünüm:


,acoustic,afrobeat,alt-rock,ambient,anime,black-metal,bluegrass,breakbeat,cantopop,chicago-house
acoustic,32,0,4,5,2,0,2,0,5,1
afrobeat,2,17,1,0,1,0,2,1,2,8
alt-rock,3,1,6,0,26,2,1,0,0,4
ambient,12,1,1,45,7,0,0,1,3,1
anime,5,2,10,11,19,2,0,0,0,0
black-metal,0,0,1,0,1,69,0,1,0,1
bluegrass,7,0,3,1,0,0,41,1,5,2
breakbeat,0,0,2,0,1,1,0,16,0,20
cantopop,42,1,7,2,4,0,4,0,30,0
chicago-house,0,1,1,0,0,0,0,9,0,73


...

,model,experiment,run_id,accuracy,f1_weighted,precision_weighted,recall_weighted,auc_roc
0,Naive Bayes,Kisi4_Naive_Bayes,3fbcda85d2b24d119e7cd388ea9c8fc9,0.299142,0.255803,0.309084,0.299142,0.855244


Confusion Matrix ilk 10x10 görünüm:


,acoustic,afrobeat,alt-rock,ambient,anime,black-metal,bluegrass,breakbeat,cantopop,chicago-house
acoustic,14,0,4,3,0,0,1,0,10,4
afrobeat,1,6,1,0,2,0,1,0,0,20
alt-rock,2,1,8,0,5,0,0,0,0,13
ambient,3,0,2,37,0,0,0,0,5,4
anime,0,2,8,5,2,0,0,0,2,13
black-metal,0,0,0,0,4,15,0,0,0,21
bluegrass,7,0,1,1,0,0,21,0,2,6
breakbeat,0,1,0,0,1,0,0,1,0,47
cantopop,12,0,1,0,1,0,4,0,31,2
chicago-house,0,0,1,0,0,0,0,0,0,86



Eğitim tamamlandı.
En iyi model: Random Forest Classifier


,model,experiment,run_id,accuracy,f1_weighted,precision_weighted,recall_weighted,auc_roc
2,Random Forest Classifier,Kisi4_Random_Forest_Classifier,3d6664908f2444528eb6da41f37d2eb1,0.559869,0.546891,0.564537,0.559869,0.944227
3,Gradient Boosted Trees (GBT) Classifier,Kisi4_GBT_Classifier,bdba765671eb48eb905df8981916c819,0.505926,0.502628,0.512976,0.505926,0.928316
1,Decision Tree Classifier,Kisi4_Decision_Tree_Classifier,0e331095d61d4970a5ca0738b516676b,0.480588,0.480489,0.518155,0.480588,0.899491
0,Logistic Regression,Kisi4_Logistic_Regression,0e0b046798774260a018c23786f200e3,0.466285,0.449011,0.466838,0.466285,0.919739
4,Naive Bayes,Kisi4_Naive_Bayes,3fbcda85d2b24d119e7cd388ea9c8fc9,0.299142,0.255803,0.309084,0.299142,0.855244


## Bu Notebookun Ürettiği Dosyalar

Dashboard notebooku şu dosyaları kullanır:

- `dashboard/data/model_metrics.csv`
- `dashboard/data/rf_feature_importance.csv`
- `dashboard/data/roc_curve_points.csv`
- `dashboard/data/best_confusion_matrix.csv`
- `dashboard/data/best_model_predictions.csv`
- `dashboard/data/best_model_metadata.json`

MLflow kayıtları `mlruns` altında tutulur.
